# MWAHAHA Task A EN - Ensemble Runs

Notebook operativo per generare candidate pool sequenziali con Qwen, Gemma e Ministral, poi fare ranking finale con Qwen judge.

Usa questo notebook quando puoi caricare un solo modello alla volta in LM Studio.


In [1]:
from pathlib import Path
import json
import re
import subprocess
from collections import Counter
from tqdm.auto import tqdm

ROOT = Path(r"E:/LLM")
INPUT = ROOT / "data" / "input" / "task-a-en.tsv"
POOL_DIR = ROOT / "candidate_pools" / "ensemble_v1"
ENSEMBLE_OUTPUT = ROOT / "submission" / "task-a-en.ensemble.tsv"
ENSEMBLE_DIAG = ROOT / "diagnostics_ensemble"
BASE_URL = "http://localhost:1234/v1"
PYTHON = "python"
SCRIPT = ROOT / "mwahaha_task_a_en.py"

MODELS = [
    {"alias": "qwen3-14b", "model": "qwen/qwen3-14b"},
    {"alias": "gemma-12b-qat", "model": "google/gemma-4-12b-qat"},
    {"alias": "ministral-3-14b-reasoning", "model": "mistralai/ministral-3-14b-reasoning"},
]

VARIANTS_PER_ENTRY = 3
TIMEOUT = 300
RERANK_TOP_K = 6


e:\LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def count_input_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8-sig") as f:
        return max(0, sum(1 for _ in f) - 1)


def run_with_progress(cmd, total=None, label="run", verbose=False):
    print(" ".join(map(str, cmd)))
    pattern = re.compile(r"^\[(\d+)/(\d+)\]\s*(.*)$")
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    bar = tqdm(total=total, desc=label, dynamic_ncols=True)
    recent_lines = []

    for line in proc.stdout:
        line = line.rstrip()
        recent_lines.append(line)
        recent_lines = recent_lines[-30:]

        match = pattern.match(line)
        if match:
            current = int(match.group(1))
            if total is None and bar.total is None:
                bar.total = int(match.group(2))
            if current > bar.n:
                bar.update(current - bar.n)
            detail = match.group(3).strip()
            if detail:
                bar.set_postfix_str(detail[:80])
            if verbose:
                tqdm.write(line)
        elif verbose:
            tqdm.write(line)

    code = proc.wait()
    bar.close()

    if code != 0:
        print("\nLast process output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {code}")

    return recent_lines


def pool_line_count(pool_dir=POOL_DIR):
    total = 0
    by_file = {}
    if not pool_dir.exists():
        return total, by_file
    for path in sorted(pool_dir.glob("*.jsonl")):
        count = sum(1 for line in path.open("r", encoding="utf-8-sig") if line.strip())
        by_file[path.name] = count
        total += count
    return total, by_file


TOTAL_ROWS = count_input_rows(INPUT)
TOTAL_ROWS



300

## 1. Generazione candidati con Qwen

Carica Qwen in LM Studio prima di eseguire questa cella.


In [ ]:
m = MODELS[0]
cmd = [
    PYTHON, SCRIPT, "generate-candidates",
    "--input", INPUT,
    "--pool-dir", POOL_DIR,
    "--backend", "openai",
    "--base-url", BASE_URL,
    "--model", m["model"],
    "--model-alias", m["alias"],
    "--variants-per-entry", VARIANTS_PER_ENTRY,
    "--timeout", TIMEOUT,
    "--resume",
]
run_with_progress(cmd, total=TOTAL_ROWS, label=f"generate {m['alias']}")
pool_line_count()


## 2. Generazione candidati con Gemma

Carica Gemma in LM Studio prima di eseguire questa cella. Aggiorna `MODELS[1]["model"]` se il nome esposto da LM Studio ? diverso.


In [4]:
m = MODELS[1]
cmd = [
    PYTHON, SCRIPT, "generate-candidates",
    "--input", INPUT,
    "--pool-dir", POOL_DIR,
    "--backend", "openai",
    "--base-url", BASE_URL,
    "--model", m["model"],
    "--model-alias", m["alias"],
    "--variants-per-entry", VARIANTS_PER_ENTRY,
    "--timeout", TIMEOUT,
    "--resume",
]
run_with_progress(cmd, total=TOTAL_ROWS, label=f"generate {m['alias']}")
pool_line_count()


python E:\LLM\mwahaha_task_a_en.py generate-candidates --input E:\LLM\data\input\task-a-en.tsv --pool-dir E:\LLM\candidate_pools\ensemble_v1 --backend openai --base-url http://localhost:1234/v1 --model google/gemma-4-12b-qat --model-alias gemma-12b-qat --variants-per-entry 3 --timeout 300 --resume


generate gemma-12b-qat: 100%|██████████| 300/300 [41:28<00:00,  8.30s/it, generating candidates for en_2300 with gemma-12b-qat]


(2700,
 {'gemma-12b-qat.jsonl': 900,
  'ministral-3-14b-reasoning.jsonl': 900,
  'qwen3-14b.jsonl': 900})

## 3. Generazione candidati con Ministral

Carica Ministral in LM Studio prima di eseguire questa cella. Aggiorna `MODELS[2]["model"]` se necessario.


In [ ]:
m = MODELS[2]
cmd = [
    PYTHON, SCRIPT, "generate-candidates",
    "--input", INPUT,
    "--pool-dir", POOL_DIR,
    "--backend", "openai",
    "--base-url", BASE_URL,
    "--model", m["model"],
    "--model-alias", m["alias"],
    "--variants-per-entry", VARIANTS_PER_ENTRY,
    "--timeout", TIMEOUT,
    "--resume",
]
run_with_progress(cmd, total=TOTAL_ROWS, label=f"generate {m['alias']}")
pool_line_count()


## 4. Ranking finale con Qwen judge

Ricarica Qwen in LM Studio prima di eseguire questa cella.


In [6]:
cmd = [
    PYTHON, SCRIPT, "rank-candidates",
    "--input", INPUT,
    "--pool-dir", POOL_DIR,
    "--output", ENSEMBLE_OUTPUT,
    "--backend", "openai",
    "--base-url", BASE_URL,
    "--model", MODELS[0]["model"],
    "--rerank-top-k", RERANK_TOP_K,
    "--timeout", TIMEOUT,
    "--diagnostics-dir", ENSEMBLE_DIAG,
    "--humor-model", "Humor-Research/humor-detection-comb-23",
    "--humor-model", "mohameddhiab/humor-no-humor",
    "--humor-weight", "0.20",
    "--humor-device", "-1",
    "--resume",
]
run_with_progress(cmd, total=TOTAL_ROWS, label="rank candidates")


python E:\LLM\mwahaha_task_a_en.py rank-candidates --input E:\LLM\data\input\task-a-en.tsv --pool-dir E:\LLM\candidate_pools\ensemble_v1 --output E:\LLM\submission\task-a-en.ensemble_gemma_fixed.tsv --backend openai --base-url http://localhost:1234/v1 --model qwen/qwen3-14b --rerank-top-k 6 --timeout 300 --diagnostics-dir E:\LLM\diagnostics_ensemble --humor-model Humor-Research/humor-detection-comb-23 --humor-model mohameddhiab/humor-no-humor --humor-weight 0.20 --humor-device -1 --resume


rank candidates: 100%|██████████| 300/300 [3:59:59<00:00, 48.00s/it, ranking en_2300 (9 candidates)]  


['[272/300] ranking en_2272 (9 candidates)',
 '[273/300] ranking en_2273 (9 candidates)',
 '[274/300] ranking en_2274 (9 candidates)',
 '[275/300] ranking en_2275 (9 candidates)',
 '[276/300] ranking en_2276 (9 candidates)',
 '[277/300] ranking en_2277 (9 candidates)',
 '[278/300] ranking en_2278 (9 candidates)',
 '[279/300] ranking en_2279 (9 candidates)',
 '[280/300] ranking en_2280 (9 candidates)',
 '[281/300] ranking en_2281 (9 candidates)',
 '[282/300] ranking en_2282 (9 candidates)',
 '[283/300] ranking en_2283 (9 candidates)',
 '[284/300] ranking en_2284 (9 candidates)',
 '[285/300] ranking en_2285 (9 candidates)',
 '[286/300] ranking en_2286 (9 candidates)',
 '[287/300] ranking en_2287 (9 candidates)',
 '[288/300] ranking en_2288 (9 candidates)',
 '[289/300] ranking en_2289 (9 candidates)',
 '[290/300] ranking en_2290 (9 candidates)',
 '[291/300] ranking en_2291 (9 candidates)',
 '[292/300] ranking en_2292 (9 candidates)',
 '[293/300] ranking en_2293 (9 candidates)',
 '[294/300

## 5. Validazione


In [7]:
cmd = [PYTHON, SCRIPT, "validate", "--input", INPUT, "--output", ENSEMBLE_OUTPUT]
run_with_progress(cmd, label="validate")


python E:\LLM\mwahaha_task_a_en.py validate --input E:\LLM\data\input\task-a-en.tsv --output E:\LLM\submission\task-a-en.ensemble_gemma_fixed.tsv


validate: 0it [00:01, ?it/s]


['Validation OK.']

## 6. Riepilogo candidate pool e winner per modello


In [8]:
total, by_file = pool_line_count()
print("pool records:", total)
print(by_file)

winners = Counter()
if ENSEMBLE_DIAG.exists():
    for path in ENSEMBLE_DIAG.glob("*.json"):
        data = json.loads(path.read_text(encoding="utf-8"))
        winner = data.get("winner", "")
        for cand in data.get("candidates", []):
            if cand.get("text") == winner:
                winners[cand.get("source_model") or "unknown"] += 1
                break
winners


pool records: 2700
{'gemma-12b-qat.jsonl': 900, 'ministral-3-14b-reasoning.jsonl': 900, 'qwen3-14b.jsonl': 900}


Counter({'ministral-3-14b-reasoning': 114,
         'gemma-12b-qat': 98,
         'qwen3-14b': 88})